# 第002/136节 · 本地Notebook状态与预测问题
依赖 torch，CPU基线，种子42，4个合成样本。
从仓库根目录启动 python -m jupyterlab，打开本文件并选择已安装torch的本机内核。
先从头运行全部，再按各节提示修改与核对。无需导入lesson.py。
验证范围见validation.json。提交文件保持输出为空，请在自己的本机重启并运行。


## 1. 准备可复现的本机CPU环境
按顺序执行，检查变量来自前面的单元格。


In [ ]:
import torch

torch.manual_seed(42)
torch.set_num_threads(1)
device = torch.device("cpu")
print("PyTorch", torch.__version__, "device", device)


## 2. 数据、模型与损失
按顺序执行，检查变量来自前面的单元格。


In [ ]:
# 只在本单元格修改目标规则；修改后需要重跑下游单元格。
TARGET_WEIGHT = 2.0
TARGET_BIAS = 1.0
x = torch.tensor([0., 1., 2., 3.], device=device).reshape(-1, 1)
y = TARGET_WEIGHT * x + TARGET_BIAS


def predict(features, weight, bias):
    return weight * features + bias


def mse(predictions, labels):
    # 避免(4,)和(4,1)广播，掩盖预测问题的错误。
    assert predictions.shape == labels.shape
    return torch.mean((predictions - labels) ** 2).item()


print("shapes", tuple(x.shape), tuple(y.shape))
print("labels", y.flatten().tolist())


## 3. 已保存的预测不会因参数重赋值自动变化
观察stale与recomputed列。改变参数没有重新执行predict时，保存的张量仍是上次结果。


In [ ]:
weight, bias = 1.0, 0.0
saved_prediction = predict(x, weight, bias)
saved_copy = saved_prediction.clone()
old_loss = mse(saved_prediction, y)
weight, bias = 2.0, 1.0
stale_loss = mse(saved_prediction, y)
fresh_prediction = predict(x, weight, bias)
fresh_loss = mse(fresh_prediction, y)
assert torch.equal(saved_prediction, saved_copy)
assert old_loss == stale_loss
print(f"before {old_loss:.6f}; stale {stale_loss:.6f}; recomputed {fresh_loss:.6f}")


## 4. 在同一数据上比较候选模型
按顺序执行，检查变量来自前面的单元格。


In [ ]:
# 这里只比较预设模型；没有执行梯度训练，也没有评估真实泛化。
candidates = [
    ("constant_mean", 0.0, y.mean().item()),
    ("x", 1.0, 0.0),
    ("2x+1", 2.0, 1.0),
    ("3x-1", 3.0, -1.0),
]
scores = []
for name, candidate_weight, candidate_bias in candidates:
    value = mse(predict(x, candidate_weight, candidate_bias), y)
    scores.append((name, value))
    print(f"{name}: MSE={value:.6f}")
best_name, best_loss = min(scores, key=lambda item: item[1])
assert best_loss < 1e-6
print("best", best_name)
print("PASS")


## 修改与重启（必做）
1. 记录原始输出：stale=7.5，recomputed=0；候选MSE依次为5、7.5、0、1.5。
2. 按homework.txt改变目标规则，然后从数据单元格开始重跑后续代码。
3. 使用Kernel菜单的重启内核并运行全部命令，比较结果。菜单名称可能随界面语言变化。
4. 保存练习副本与自己的观察。参考答案单独放在answers.html。
